## TASK 1 : Install Dependencies & Run Spark Session

In [2]:
# Install pyspark
! pip install pyspark

In [3]:
# Create a sparksession
from pyspark.sql import SparkSession
spark = SparkSession.builder.appName("spark").getOrCreate()

## TASK 2: Clone & Explore dataset

In [4]:
# Clone the diabetes dataset from the github repository
! git clone https://github.com/education454/diabetes_dataset

Cloning into 'diabetes_dataset'...
remote: Enumerating objects: 6, done.
remote: Counting objects: 100% (6/6), done.
remote: Compressing objects: 100% (5/5), done.
remote: Total 6 (delta 0), reused 0 (delta 0), pack-reused 0 (from 0)
Receiving objects: 100% (6/6), 13.02 KiB | 350.00 KiB/s, done.


In [5]:
# Check if the dataset exists
! ls diabetes_dataset

diabetes.csv  new_test.csv


In [7]:
# Create spark dataframe
df = spark.read.csv("/content/diabetes_dataset/diabetes.csv", header = True, inferSchema = True)

In [8]:
# Display the dataframe
df.show()

+-----------+-------+-------------+-------------+-------+----+------------------------+---+-------+
|Pregnancies|Glucose|BloodPressure|SkinThickness|Insulin| BMI|DiabetesPedigreeFunction|Age|Outcome|
+-----------+-------+-------------+-------------+-------+----+------------------------+---+-------+
|          2|    138|           62|           35|      0|33.6|                   0.127| 47|      1|
|          0|     84|           82|           31|    125|38.2|                   0.233| 23|      0|
|          0|    145|            0|            0|      0|44.2|                    0.63| 31|      1|
|          0|    135|           68|           42|    250|42.3|                   0.365| 24|      1|
|          1|    139|           62|           41|    480|40.7|                   0.536| 21|      0|
|          0|    173|           78|           32|    265|46.5|                   1.159| 58|      0|
|          4|     99|           72|           17|      0|25.6|                   0.294| 28|      0|


In [9]:
# Print the schema
df.printSchema()

root
 |-- Pregnancies: integer (nullable = true)
 |-- Glucose: integer (nullable = true)
 |-- BloodPressure: integer (nullable = true)
 |-- SkinThickness: integer (nullable = true)
 |-- Insulin: integer (nullable = true)
 |-- BMI: double (nullable = true)
 |-- DiabetesPedigreeFunction: double (nullable = true)
 |-- Age: integer (nullable = true)
 |-- Outcome: integer (nullable = true)



In [11]:
# Count the total no. of diabetic and non-diabetic class
print((df.count(), len(df.columns)))
df.groupBy('Outcome').count().show()

(2000, 9)
+-------+-----+
|Outcome|count|
+-------+-----+
|      1|  684|
|      0| 1316|
+-------+-----+



In [12]:
# Get the summary statistics
df.describe().show()

+-------+-----------------+------------------+------------------+-----------------+-----------------+------------------+------------------------+------------------+------------------+
|summary|      Pregnancies|           Glucose|     BloodPressure|    SkinThickness|          Insulin|               BMI|DiabetesPedigreeFunction|               Age|           Outcome|
+-------+-----------------+------------------+------------------+-----------------+-----------------+------------------+------------------------+------------------+------------------+
|  count|             2000|              2000|              2000|             2000|             2000|              2000|                    2000|              2000|              2000|
|   mean|           3.7035|          121.1825|           69.1455|           20.935|           80.254|32.192999999999984|     0.47092999999999974|           33.0905|             0.342|
| stddev|3.306063032730656|32.068635649902916|19.188314815604098|16.103242909926

## TASK 3: Data Cleaning & Preparation

In [14]:
# Check for null values
for col in df.columns:
  print(col + ":", df[df[col].isNull()].count())

Pregnancies: 0
Glucose: 0
BloodPressure: 0
SkinThickness: 0
Insulin: 0
BMI: 0
DiabetesPedigreeFunction: 0
Age: 0
Outcome: 0


In [15]:
# Look for the unnecessary values present
def count_zeros():
  columns_list = ['Glucose', 'BloodPressure', 'SkinThickness', 'Insulin', 'BMI']
  for i in columns_list:
    print(i + ":", df[df[i] == 0].count())

In [16]:
count_zeros()

Glucose: 13
BloodPressure: 90
SkinThickness: 573
Insulin: 956
BMI: 28


In [18]:
# Calculate and replace the unnecessary values by the mean value
from pyspark.sql.functions import *
for i in df.columns[1:6]:
  data = df.agg({i:'mean'}).first()[0]
  print("Mean Value for {} is {}".format(i, int(data)))
  df = df.withColumn(i, when(df[i] == 0, int(data)).otherwise(df[i]))

Mean Value for Glucose is 121
Mean Value for BloodPressure is 69
Mean Value for SkinThickness is 20
Mean Value for Insulin is 80
Mean Value for BMI is 32


In [19]:
# Display the dataframe
df.show()

+-----------+-------+-------------+-------------+-------+----+------------------------+---+-------+
|Pregnancies|Glucose|BloodPressure|SkinThickness|Insulin| BMI|DiabetesPedigreeFunction|Age|Outcome|
+-----------+-------+-------------+-------------+-------+----+------------------------+---+-------+
|          2|    138|           62|           35|     80|33.6|                   0.127| 47|      1|
|          0|     84|           82|           31|    125|38.2|                   0.233| 23|      0|
|          0|    145|           69|           20|     80|44.2|                    0.63| 31|      1|
|          0|    135|           68|           42|    250|42.3|                   0.365| 24|      1|
|          1|    139|           62|           41|    480|40.7|                   0.536| 21|      0|
|          0|    173|           78|           32|    265|46.5|                   1.159| 58|      0|
|          4|     99|           72|           17|     80|25.6|                   0.294| 28|      0|


## TASK 4: Correlation Analysis & Feature Selection

In [21]:
# Find the correlation among the set of input & output variables
for i in df.columns:
  print(":Correlation to outcome for {} is {}".format(i, df.stat.corr('Outcome', i)))

:Correlation to outcome for Pregnancies is 0.22443699263363961
:Correlation to outcome for Glucose is 0.48796646527321064
:Correlation to outcome for BloodPressure is 0.17171333286446713
:Correlation to outcome for SkinThickness is 0.1659010662889893
:Correlation to outcome for Insulin is 0.1711763270226193
:Correlation to outcome for BMI is 0.2827927569760082
:Correlation to outcome for DiabetesPedigreeFunction is 0.1554590791569403
:Correlation to outcome for Age is 0.23650924717620253
:Correlation to outcome for Outcome is 1.0


In [23]:
# Feature selection
from pyspark.ml.feature import VectorAssembler
assembler = VectorAssembler(inputCols = ['Pregnancies', 'Glucose', 'BloodPressure', 'SkinThickness', 'Insulin', 'BMI', 'DiabetesPedigreeFunction', 'Age'], outputCol = 'Features')
output_data = assembler.transform(df)

In [24]:
# Print the schema
output_data.printSchema()

root
 |-- Pregnancies: integer (nullable = true)
 |-- Glucose: integer (nullable = true)
 |-- BloodPressure: integer (nullable = true)
 |-- SkinThickness: integer (nullable = true)
 |-- Insulin: integer (nullable = true)
 |-- BMI: double (nullable = true)
 |-- DiabetesPedigreeFunction: double (nullable = true)
 |-- Age: integer (nullable = true)
 |-- Outcome: integer (nullable = true)
 |-- Features: vector (nullable = true)



In [27]:
# Display dataframe
output_data.show()

+-----------+-------+-------------+-------------+-------+----+------------------------+---+-------+--------------------+
|Pregnancies|Glucose|BloodPressure|SkinThickness|Insulin| BMI|DiabetesPedigreeFunction|Age|Outcome|            Features|
+-----------+-------+-------------+-------------+-------+----+------------------------+---+-------+--------------------+
|          2|    138|           62|           35|     80|33.6|                   0.127| 47|      1|[2.0,138.0,62.0,3...|
|          0|     84|           82|           31|    125|38.2|                   0.233| 23|      0|[0.0,84.0,82.0,31...|
|          0|    145|           69|           20|     80|44.2|                    0.63| 31|      1|[0.0,145.0,69.0,2...|
|          0|    135|           68|           42|    250|42.3|                   0.365| 24|      1|[0.0,135.0,68.0,4...|
|          1|    139|           62|           41|    480|40.7|                   0.536| 21|      0|[1.0,139.0,62.0,4...|
|          0|    173|           

## TASK 5: Split Dataset & Build the Model

In [28]:
# Create final data
from pyspark.ml.classification import LogisticRegression
final_data = output_data.select('Features', 'Outcome')

In [29]:
# Print schema of final data
final_data.printSchema()

root
 |-- Features: vector (nullable = true)
 |-- Outcome: integer (nullable = true)



In [31]:
# Split the dataset ; build the model
train, test = final_data.randomSplit([0.7, 0.3])
models = LogisticRegression(labelCol = 'Outcome')
model = models.fit(train)

In [32]:
# Summary of the model
summary = model.summary
summary.predictions.describe().show()

+-------+-------------------+-------------------+
|summary|            Outcome|         prediction|
+-------+-------------------+-------------------+
|  count|               1397|               1397|
|   mean| 0.3457408732999284| 0.2634216177523264|
| stddev|0.47577952789741335|0.44064686485175947|
|    min|                0.0|                0.0|
|    max|                1.0|                1.0|
+-------+-------------------+-------------------+



## TASK 6: Evaluate and Save the Model

In [33]:
from pyspark.ml.evaluation import BinaryClassificationEvaluator
predictions = model.evaluate(test)

In [34]:
predictions.predictions.show(20)

+--------------------+-------+--------------------+--------------------+----------+
|            Features|Outcome|       rawPrediction|         probability|prediction|
+--------------------+-------+--------------------+--------------------+----------+
|[0.0,73.0,69.0,20...|      0|[3.98047001961491...|[0.98166557076137...|       0.0|
|[0.0,84.0,64.0,22...|      0|[2.39608227360134...|[0.91652806901148...|       0.0|
|[0.0,84.0,82.0,31...|      0|[2.45259975316424...|[0.92075135799412...|       0.0|
|[0.0,86.0,68.0,32...|      0|[2.49431341959681...|[0.92374220680128...|       0.0|
|[0.0,91.0,68.0,32...|      0|[2.16356262091154...|[0.89692936848193...|       0.0|
|[0.0,91.0,68.0,32...|      0|[2.16356262091154...|[0.89692936848193...|       0.0|
|[0.0,91.0,80.0,20...|      0|[2.27411900750795...|[0.90671078368551...|       0.0|
|[0.0,93.0,60.0,20...|      0|[2.29718859706971...|[0.90864393200466...|       0.0|
|[0.0,93.0,60.0,20...|      0|[2.29718859706971...|[0.90864393200466...|    

In [36]:
evaluator = BinaryClassificationEvaluator(rawPredictionCol = 'rawPrediction', labelCol = 'Outcome')
evaluator.evaluate(model.transform(test))

0.8537536199599021

In [37]:
# Save model
model.save("Model")

In [38]:
# Load saved model back to the environment
from pyspark.ml.classification import LogisticRegressionModel
model = LogisticRegressionModel.load("Model")

## TASK 7: Prediction on New Data with the saved model

In [39]:
# Create a new spark dataframe
test_df = spark.read.csv("/content/diabetes_dataset/new_test.csv", header = True, inferSchema = True)

In [40]:
# Print the schema
test_df.printSchema()

root
 |-- Pregnancies: integer (nullable = true)
 |-- Glucose: integer (nullable = true)
 |-- BloodPressure: integer (nullable = true)
 |-- SkinThickness: integer (nullable = true)
 |-- Insulin: integer (nullable = true)
 |-- BMI: double (nullable = true)
 |-- DiabetesPedigreeFunction: double (nullable = true)
 |-- Age: integer (nullable = true)



In [41]:
# Create an additional feature merged column
test_data = assembler.transform(test_df)

In [42]:
# Print the schema
test_data.printSchema()

root
 |-- Pregnancies: integer (nullable = true)
 |-- Glucose: integer (nullable = true)
 |-- BloodPressure: integer (nullable = true)
 |-- SkinThickness: integer (nullable = true)
 |-- Insulin: integer (nullable = true)
 |-- BMI: double (nullable = true)
 |-- DiabetesPedigreeFunction: double (nullable = true)
 |-- Age: integer (nullable = true)
 |-- Features: vector (nullable = true)



In [43]:
# Use model to make predictions
results = model.transform(test_data)
results.printSchema()

root
 |-- Pregnancies: integer (nullable = true)
 |-- Glucose: integer (nullable = true)
 |-- BloodPressure: integer (nullable = true)
 |-- SkinThickness: integer (nullable = true)
 |-- Insulin: integer (nullable = true)
 |-- BMI: double (nullable = true)
 |-- DiabetesPedigreeFunction: double (nullable = true)
 |-- Age: integer (nullable = true)
 |-- Features: vector (nullable = true)
 |-- rawPrediction: vector (nullable = true)
 |-- probability: vector (nullable = true)
 |-- prediction: double (nullable = false)



In [45]:
# Display the predictions
results.select('features', 'prediction').show()

+--------------------+----------+
|            features|prediction|
+--------------------+----------+
|[1.0,190.0,78.0,3...|       1.0|
|[0.0,80.0,84.0,36...|       0.0|
|[2.0,138.0,82.0,4...|       1.0|
|[1.0,110.0,63.0,4...|       1.0|
+--------------------+----------+

